In [5]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# TimeGAN Financial Fraud Detection: End-to-End Pipeline\n",
    "Trains TimeGAN on Google Colab GPU, generates synthetic temporal fraud data, and evaluates via TSTR and DCR."
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### 1. Verify GPU Hardware Acceleration"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import torch\n",
    "print(\"PyTorch Version:\", torch.__version__)\n",
    "if torch.cuda.is_available():\n",
    "    print(\"Active GPU:\", torch.cuda.get_device_name(0))\n",
    "else:\n",
    "    print(\"WARNING: GPU not active. Go to Runtime -> Change runtime type -> Select T4 GPU.\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### 2. Mount Google Drive for Persistent Storage"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "from google.colab import drive\n",
    "import os\n",
    "drive.mount('/content/drive')\n",
    "\n",
    "DRIVE_DIR = '/content/drive/MyDrive/TimeGAN_FYP_Outputs'\n",
    "os.makedirs(f\"{DRIVE_DIR}/checkpoints\", exist_ok=True)\n",
    "os.makedirs(f\"{DRIVE_DIR}/synthetic\", exist_ok=True)\n",
    "os.makedirs(f\"{DRIVE_DIR}/figures\", exist_ok=True)\n",
    "print(\"Persistent directory ready at:\", DRIVE_DIR)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### 3. Clone Repository & Install Dependencies"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import os\n",
    "REPO_NAME = \"TIMEGAN-SyntheticData-FinancialFraudDetection\"\n",
    "GITHUB_USERNAME = \"syaahiddd\"\n",
    "REPO_URL = f\"https://github.com/{GITHUB_USERNAME}/{REPO_NAME}.git\"\n",
    "\n",
    "if not os.path.exists(REPO_NAME):\n",
    "    !git clone {REPO_URL}\n",
    "    %cd {REPO_NAME}\n",
    "else:\n",
    "    %cd {REPO_NAME}\n",
    "    !git pull\n",
    "\n",
    "!pip install -r requirements.txt -q\n",
    "print(\"Environment initialized.\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### 4. Fetch Kaggle Dataset into Colab"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import os\n",
    "from google.colab import files\n",
    "\n",
    "if not os.path.exists(\"data/raw/creditcard.csv\"):\n",
    "    print(\"Upload kaggle.json to download the dataset:\")\n",
    "    if not os.path.exists(\"/root/.kaggle/kaggle.json\"):\n",
    "        uploaded = files.upload()\n",
    "        !mkdir -p /root/.kaggle\n",
    "        !mv kaggle.json /root/.kaggle/\n",
    "        !chmod 600 /root/.kaggle/kaggle.json\n",
    "    \n",
    "    !kaggle datasets download -d mlg-ulb/creditcardfraud -p data/raw\n",
    "    !unzip -q data/raw/creditcardfraud.zip -d data/raw\n",
    "    print(\"creditcard.csv ready.\")\n",
    "else:\n",
    "    print(\"Dataset already exists in data/raw/.\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### 5. Ingest & Window Sequences"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import sys\n",
    "sys.path.append('.')\n",
    "import numpy as np\n",
    "from src.data_loader import load_and_preprocess_creditcard\n",
    "\n",
    "SEQ_LEN = 24\n",
    "DATA_PATH = \"data/raw/creditcard.csv\"\n",
    "\n",
    "sequences, window_labels, loader = load_and_preprocess_creditcard(\n",
    "    csv_path=DATA_PATH,\n",
    "    seq_len=SEQ_LEN,\n",
    "    stride=2,\n",
    "    subsample_limit=25000\n",
    ")\n",
    "\n",
    "split_idx = int(len(sequences) * 0.8)\n",
    "real_train_x, real_test_x = sequences[:split_idx], sequences[split_idx:]\n",
    "real_train_y, real_test_y = window_labels[:split_idx], window_labels[split_idx:]\n",
    "print(f\"Train shape: {real_train_x.shape}, Test shape: {real_test_x.shape}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### 6. Train TimeGAN Model on GPU"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "from src.timegan import TimeGAN, get_device\n",
    "\n",
    "device = get_device()\n",
    "print(\"Training on device:\", device)\n",
    "\n",
    "N_FEATURES = real_train_x.shape[2]\n",
    "model = TimeGAN(\n",
    "    feature_dim=N_FEATURES,\n",
    "    seq_len=SEQ_LEN,\n",
    "    hidden_dim=24,\n",
    "    num_layers=3,\n",
    "    lr=1e-3,\n",
    "    device=device\n",
    ")\n",
    "\n",
    "history = model.fit(\n",
    "    real_train_x,\n",
    "    batch_size=128,\n",
    "    pretrain_epochs=30,\n",
    "    joint_epochs=60\n",
    ")\n",
    "\n",
    "import torch\n",
    "torch.save(model.generator.state_dict(), f\"{DRIVE_DIR}/checkpoints/generator.pt\")\n",
    "torch.save(model.recovery.state_dict(), f\"{DRIVE_DIR}/checkpoints/recovery.pt\")\n",
    "print(\"Checkpoints saved to Google Drive.\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### 7. Generate Synthetic Sequences"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "n_gen = len(real_train_x)\n",
    "synth_sequences = model.sample(num_samples=n_gen)\n",
    "np.save(f\"{DRIVE_DIR}/synthetic/synthetic_sequences.npy\", synth_sequences)\n",
    "print(f\"Generated and saved {synth_sequences.shape[0]} sequences.\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### 8. Fidelity & Privacy Metrics"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "from src.metrics import evaluate_distribution_overlap, compute_discriminative_score, compute_dcr_metrics\n",
    "\n",
    "fig_path = f\"{DRIVE_DIR}/figures/pca_tsne_comparison.png\"\n",
    "evaluate_distribution_overlap(real_train_x, synth_sequences, n_samples=1000, save_path=fig_path)\n",
    "\n",
    "disc_score = compute_discriminative_score(real_train_x, synth_sequences, epochs=15, batch_size=128, device=device)\n",
    "print(f\"Post-Hoc Discriminative Score (|Acc - 0.5|): {disc_score:.4f}\")\n",
    "\n",
    "dcr_stats = compute_dcr_metrics(real_train_x, synth_sequences, real_test_x, n_sample=500)\n",
    "print(\"DCR Privacy Metrics:\")\n",
    "for k, v in dcr_stats.items():\n",
    "    print(f\"  {k}: {v:.4f}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### 9. Downstream Fraud Utility (TSTR vs TRTR)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "from src.downstream import DownstreamBenchmark\n",
    "\n",
    "benchmark = DownstreamBenchmark(random_state=42)\n",
    "synth_labels = real_train_y.copy()\n",
    "\n",
    "results_df = benchmark.evaluate_tstr_vs_trtr(\n",
    "    real_train_x, real_train_y,\n",
    "    synth_sequences, synth_labels,\n",
    "    real_test_x, real_test_y,\n",
    "    repr_method=\"summary\"\n",
    ")\n",
    "\n",
    "print(\"\\nTSTR vs TRTR Benchmark Table:\")\n",
    "display(results_df)\n",
    "results_df.to_csv(f\"{DRIVE_DIR}/benchmark_results.csv\", index=False)\n",
    "print(\"Saved results table to Google Drive.\")"
   ]
  }
 ],
 "metadata": {
  "language_info": {
   "name": "python"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 2
}

NameError: name 'null' is not defined

# TimeGAN Financial Fraud Detection - Training Pipeline (Google Colab)
This notebook connects to your GitHub repository, mounts Google Drive for persistent checkpoints, and handles GPU-accelerated training.

### 1. Mount Google Drive
Persistent storage for model checkpoints (`.pt`), synthetic outputs, and evaluation metrics across Colab session disconnects.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
# Create an external checkpoint backup folder in your Google Drive
DRIVE_PROJECT_PATH = '/content/drive/MyDrive/FYP_TimeGAN_Checkpoints'
os.makedirs(DRIVE_PROJECT_PATH, exist_ok=True)
print(f"Persistent Drive directory ready at: {DRIVE_PROJECT_PATH}")

Mounted at /content/drive
Persistent Drive directory ready at: /content/drive/MyDrive/FYP_TimeGAN_Checkpoints


### 2. Verify GPU Hardware Acceleration

In [2]:
import torch

print("PyTorch Version:", torch.__version__)
cuda_available = torch.cuda.is_available()
print("CUDA Available:", cuda_available)

if cuda_available:
    print("Active GPU:", torch.cuda.get_device_name(0))
    print("Total Memory (GB):", round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2))
else:
    print("WARNING: No GPU detected. Go to Runtime -> Change runtime type -> Select T4 GPU.")

PyTorch Version: 2.11.0+cpu
CUDA Available: False


### 3. Clone / Pull Repository & Install Dependencies
For private repositories, use a GitHub Personal Access Token (PAT).

In [3]:
# Set your GitHub credentials here
GITHUB_USERNAME = "<your-username>"
GITHUB_TOKEN = "<your-personal-access-token>"  # Leave blank if repository is Public
REPO_NAME = "TIMEGAN-SyntheticData-FinancialFraudDetection"

import os
if not os.path.exists(REPO_NAME):
    if GITHUB_TOKEN and GITHUB_TOKEN != "<your-personal-access-token>":
        repo_url = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"
    else:
        repo_url = f"https://github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"
    !git clone {repo_url}
else:
    print("Repository already exists. Pulling latest commits...")
    %cd {REPO_NAME}
    !git pull
    %cd ..

# Navigate into repo and install packages
%cd {REPO_NAME}
!pip install -r requirements.txt -q
print("Dependencies installed successfully.")

/bin/bash: line 1: your-username: No such file or directory
[Errno 2] No such file or directory: 'TIMEGAN-SyntheticData-FinancialFraudDetection'
/content
ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'
Dependencies installed successfully.


### 4. Test Modular Import
Ensure `src` modules are recognized by Colab's Python path.

In [4]:
import sys
sys.path.append('.')

# Verify that data_loader imports correctly
from src.data_loader import FinancialTimeSeriesLoader, generate_mock_financial_data

loader = FinancialTimeSeriesLoader(seq_len=24)
print("Successfully imported FinancialTimeSeriesLoader into Colab runtime!")

ModuleNotFoundError: No module named 'src'